In [4]:
import pandas as pd
import numpy as np
import hashlib
import os
import tldextract
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/url_cleaned.csv")
assert not df["URL"].duplicated().any()
print(len(df), df["label"].value_counts().to_dict())

235370 {0: 134850, 1: 100520}


In [5]:
extract = tldextract.TLDExtract(include_psl_private_domains=True)

def reg_domain(url):
    ext = extract(url)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}".lower()
    return (ext.domain or url).lower()   # raw IPs, localhost, malformed URLs

df["reg_domain"] = df["URL"].map(reg_domain)

dom = df.groupby("reg_domain")["label"].agg(n="size", phish_frac="mean")
print("Unique registered domains:", len(dom))
print("Domains with both labels:", ((dom.phish_frac > 0) & (dom.phish_frac < 1)).sum())
dom.sort_values("n", ascending=False).head(15)

Unique registered domains: 198052
Domains with both labels: 106


,n,phish_frac
reg_domain,,
weeblysite.com,3079,1.000000
ipfs.io,1552,1.000000
xsph.ru,928,1.000000
google.com,782,1.000000
godaddysites.com,770,1.000000
webwave.dev,559,1.000000
weebly.com,552,1.000000
glitch.me,509,1.000000
fleek.co,468,1.000000


In [7]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42)
fold = np.empty(len(df), dtype=int)
for i, (_, idx) in enumerate(sgkf.split(df, df["label"], groups=df["reg_domain"])):
    fold[idx] = i

df["split"] = np.select([fold < 14, fold < 17], ["train", "val"], default="test")

summary = df.groupby("split").agg(rows=("URL", "size"),
                                  phish_frac=("label", "mean"),
                                  domains=("reg_domain", "nunique"))
summary["row_frac"] = summary["rows"] / len(df)
print(summary.loc[["train", "val", "test"]])

s = {k: set(df.loc[df.split == k, "reg_domain"]) for k in ["train", "val", "test"]}
assert not (s["train"] & s["val"]) and not (s["train"] & s["test"]) and not (s["val"] & s["test"])
print("No domain overlap ✓")

         rows  phish_frac  domains  row_frac
split                                       
train  164758    0.427075   138170  0.699996
val     35306    0.427066    29941  0.150002
test    35306    0.427066    29941  0.150002
No domain overlap ✓


In [11]:
os.makedirs("data/splits", exist_ok=True)
os.makedirs("data/locked", exist_ok=True)
keep = ["URL", "Domain", "reg_domain", "URLLength", "TLD", "IsHTTPS", "label"]

df.loc[df.split == "train", keep].to_csv("../data/splits/train.csv", index=False)
df.loc[df.split == "val",   keep].to_csv("../data/splits/val.csv",   index=False)

test_path = "../data/locked/test.csv"
df.loc[df.split == "test", keep].to_csv(test_path, index=False)
os.chmod(test_path, 0o444)   # read-only, so you can't overwrite it by accident
print("test.csv SHA-256:", hashlib.sha256(open(test_path, "rb").read()).hexdigest())

test.csv SHA-256: 4e6505ee517fb94a54a2622e441b4671aee7aac2a3c80146e67115a91ae22521
